# Regress?o 01 - tarefa 03 - transforma??es em X e Y

Carregue os pacotes necess?rios e a base de gorjetas.

### I. Modelo no valor da gorjeta

1. Crie a matriz de design (e a matriz y) utilizando o Patsy, para um modelo em ```tip```, explicada por ```sex, smoker, diner e net_bill```.  
2. Remova as vari?veis n?o significantes.  
3. observe o gr?fico de res?duos em fun??o de ```net_bill```  
4. teste transformar ```net_bill``` no log e um polin?mio. Escolha o melhor modelo.

In [ ]:
import pandas as pd
import seaborn as sns
from seaborn import load_dataset
import patsy
from patsy import dmatrices

import matplotlib.pyplot as plt

import numpy as np

import statsmodels.formula.api as smf
import statsmodels.api as sm

In [ ]:
tips = sns.load_dataset("tips")
tips['tip_pct'] = tips['tip'] / (tips['total_bill'] - tips['tip'])
tips['net_bill'] = tips['total_bill'] - tips['tip']
tips.head()

In [ ]:
# Ajustando o modelo inicial
model_1 = smf.ols("tip ~ sex + smoker + time + net_bill", data=tips).fit()
print("Modelo inicial:")
print(model_1.summary())

# Remo??o das vari?veis n?o significativas
formula_reduzida = "tip ~ net_bill"
y_reduzida, X_reduzida = dmatrices(formula_reduzida, data=tips, return_type='dataframe')
modelo_reduzido = sm.OLS(y_reduzida, X_reduzida).fit()



In [ ]:
# Gr?fico de res?duos em fun??o de net_bill
plt.figure(figsize=(8, 5))
plt.scatter(tips['net_bill'], model_1.resid)
plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.title('Res?duos vs. Net Bill')
plt.xlabel('Net Bill')
plt.ylabel('Res?duos')
plt.show()

In [ ]:
# Transforma??es: log e polin?mio
tips['log_net_bill'] = np.log(tips['net_bill'] + 1)  # +1 para evitar log(0)
tips['net_bill_squared'] = tips['net_bill'] ** 2

# Ajustando modelo com log
model_log = smf.ols("tip ~ sex + smoker + time + log_net_bill", data=tips).fit()
print("Modelo com transforma??o log:")
print(model_log.summary())

In [ ]:
# Ajustando modelo polinomial
model_poly = smf.ols("tip ~ sex + smoker + time + net_bill + net_bill_squared", data=tips).fit()
print("Modelo com transforma??o polinomial:")
print(model_poly.summary())

O Modelo Polinominal teve a melhor resposta.

### II. Modelo no valor do percentual da gorjeta

1. Crie a matriz de design (e a matriz y) utilizando o Patsy, para um modelo no log de ```tip```, explicado por ```sex, smoker, diner e net_bill```.
2. Remova as vari?veis n?o significantes.
3. Observe o gr?fico de res?duos em fun??o de ```net_bill```
4. Teste transformar ```net_bill``` no log e um polin?mio. Escolha o melhor modelo.
5. Do modelo final deste item, calcule o $R^2$ na escala de ```tip``` (sem o log). Compare com o modelo do item 1. Qual tem melhor coeficiente de determina??o?

In [ ]:
# Criando a matriz de design (X) e a matriz dependente (y) com Patsy
y, X = patsy.dmatrices("np.log(tip) ~ sex + smoker + time + net_bill", data=tips, return_type="dataframe")

# Ajustando o modelo inicial
model_patsy = sm.OLS(y, X).fit()
print("Modelo inicial com log(tip):")
print(model_patsy.summary())

# Identificar vari?veis significativas e refazer o modelo
# Removendo vari?veis n?o significantes (p > 0.05) e ajustando o modelo novamente
y, X_significant = patsy.dmatrices("np.log(tip) ~ net_bill", data=tips, return_type="dataframe")
model_significant = sm.OLS(y, X_significant).fit()
print("\nModelo com vari?veis significativas:")
print(model_significant.summary())


In [ ]:

# Gr?fico de res?duos em fun??o de net_bill
plt.figure(figsize=(8, 5))
plt.scatter(tips['net_bill'], model_significant.resid)
plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.title('Res?duos vs. Net Bill (log(tip))')
plt.xlabel('Net Bill')
plt.ylabel('Res?duos')
plt.show()



In [ ]:
# Testando transforma??es em net_bill
tips['log_net_bill'] = np.log(tips['net_bill'] + 1)  # +1 para evitar log(0)
tips['net_bill_squared'] = tips['net_bill'] ** 2

# Modelo com log(net_bill)
y, X_log = patsy.dmatrices("np.log(tip) ~ log_net_bill", data=tips, return_type="dataframe")
model_log = sm.OLS(y, X_log).fit()

# Modelo polinomial com net_bill
y, X_poly = patsy.dmatrices("np.log(tip) ~ net_bill + net_bill_squared", data=tips, return_type="dataframe")
model_poly = sm.OLS(y, X_poly).fit()

print("\nModelo com log(net_bill):")
print(model_log.summary())

print("\nModelo polinomial:")
print(model_poly.summary())

In [ ]:
# Previs?o na escala original (sem log)
y_pred_original_scale = np.exp(model_poly.predict(X_poly))
y_actual = tips['tip']

# R? na escala original
sst = np.sum((y_actual - y_actual.mean())**2)
ssr = np.sum((y_actual - y_pred_original_scale)**2)
r2_original_scale = 1 - (ssr / sst)

# Comparar com o modelo inicial do item 1
r2_initial = model_1.rsquared

print("\nR? na escala original (melhor modelo):", r2_original_scale)
print("R? do modelo inicial:", r2_initial)

### III. Previs?o de renda

Vamos trabalhar a base que voc? vai usar no projeto do final deste ciclo.

Carregue a base ```previsao_de_renda.csv```.

|variavel|descri??o|
|-|-|
|data_ref                | Data de refer?ncia de coleta das vari?veis |
|index                   | C?digo de identifica??o do cliente|
|sexo                    | Sexo do cliente|
|posse_de_veiculo        | Indica se o cliente possui ve?culo|
|posse_de_imovel         | Indica se o cliente possui im?vel|
|qtd_filhos              | Quantidade de filhos do cliente|
|tipo_renda              | Tipo de renda do cliente|
|educacao                | Grau de instru??o do cliente|
|estado_civil            | Estado civil do cliente|
|tipo_residencia         | Tipo de resid?ncia do cliente (pr?pria, alugada etc)|
|idade                   | Idade do cliente|
|tempo_emprego           | Tempo no emprego atual|
|qt_pessoas_residencia   | Quantidade de pessoas que moram na resid?ncia|
|renda                   | Renda em reais|

1. Ajuste um modelo de regress?o linear simples para explicar ```renda``` como vari?vel resposta, por ```tempo_emprego``` como vari?vel explicativa. Observe que h? muitas observa??es nessa tabela. Utilize os recursos que achar necess?rio.
2. Fa?a uma an?lise de res?duos. Com os recursos vistos neste m?dulo, como voc? melhoraria esta regress?o?
3. Ajuste um modelo de regress?o linear m?ltipla para explicar ```renda``` (ou uma transforma??o de ```renda```) de acordo com as demais vari?veis.
4. Remova as vari?veis n?o significantes e ajuste novamente o modelo. Interprete os par?metros
5. Fa?a uma an?lise de res?duos. Avalie a qualidade do ajuste.

In [ ]:
# 1. Carregar os dados
df = pd.read_csv("previsao_de_renda.csv")

# Explora??o inicial dos dados
print(df.info())
print(df.describe())




In [ ]:

# Verificar valores nulos e tratar se necess?rio
print("\nValores nulos por vari?vel:")
print(df.isnull().sum())


In [ ]:
# Remover valores nulos de tempo_emprego
df_clean = df.dropna(subset=['tempo_emprego'])

# Ajustar novamente o modelo simples com os dados limpos
model_simple = smf.ols("renda ~ tempo_emprego", data=df_clean).fit()

# An?lise de res?duos do modelo simples
plt.figure(figsize=(8, 5))
plt.scatter(df_clean['tempo_emprego'], model_simple.resid)
plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.title('Res?duos vs. Tempo de Emprego')
plt.xlabel('Tempo de Emprego')
plt.ylabel('Res?duos')
plt.show()

In [ ]:
# Propor melhorias (exemplo: transforma??o logar?tmica na renda)
df['log_renda'] = np.log(df['renda'] + 1)

# Novo modelo com transforma??o logar?tmica
model_log_simple = smf.ols("log_renda ~ tempo_emprego", data=df).fit()
print("\nModelo com Transforma??o Logar?tmica na Renda:")
print(model_log_simple.summary())


In [ ]:

# 3. Regress?o linear m?ltipla: renda ~ todas as vari?veis explicativas
# Corrigir nomes das colunas para evitar erros
df_dummies.columns = df_dummies.columns.str.replace(' ', '_').str.replace('?', 'a').str.replace('?', 'c').str.replace('?', 'e')

# Ajustar o modelo novamente com as vari?veis corrigidas
model_multiple = smf.ols(
    "log_renda ~ posse_de_veiculo + posse_de_imovel + qtd_filhos + idade + tempo_emprego + qt_pessoas_residencia + " +
    " + ".join([col for col in df_dummies.columns if 'sexo_' in col or
                                                    'tipo_renda_' in col or
                                                    'educacao_' in col or
                                                    'estado_civil_' in col or
                                                    'tipo_residencia_' in col]),
    data=df_dummies
).fit()

print("\nModelo de Regress?o Linear M?ltipla:")
print(model_multiple.summary())


In [ ]:
# Verificar as colunas dummies criadas
print("Colunas no DataFrame ap?s transforma??o:")
print(df_dummies.columns.tolist())

In [ ]:

# Remover vari?veis n?o significativas (p > 0.05) e ajustar novamente
# Novo modelo considerando apenas vari?veis significativas
model_final = smf.ols("log_renda ~ idade + tempo_emprego + educacao_Superior_completo + estado_civil_Uniao", data=df_dummies).fit()
print("\nModelo Final de Regress?o Linear M?ltipla:")
print(model_final.summary())


In [ ]:

# 4. An?lise de res?duos do modelo final
plt.figure(figsize=(8, 5))
plt.scatter(model_final.fittedvalues, model_final.resid)
plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.title('Res?duos vs. Valores Ajustados (Modelo Final)')
plt.xlabel('Valores Ajustados')
plt.ylabel('Res?duos')
plt.show()

# Histograma dos res?duos
plt.figure(figsize=(8, 5))
sns.histplot(model_final.resid, kde=True, bins=30)
plt.title('Histograma dos Res?duos')
plt.xlabel('Res?duos')
plt.ylabel('Frequ?ncia')
plt.show()

# Qualidade do ajuste
print("\nR? do Modelo Final:", model_final.rsquared)
print("R? Ajustado do Modelo Final:", model_final.rsquared_adj)